# QLoRA Fine-Tuning Pipeline with PyTorch & PEFT
### 4-bit Quantized Supervised Fine-Tuning (SFT) on Qwen2.5-3B in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook provides a dedicated, lightweight training and evaluation workflow:
1. **Hardware & Dependency Setup**: Installs minimal fine-tuning packages (`transformers`, `peft`, `trl`, `bitsandbytes`, `accelerate`). No vLLM or complex C++ compilation required.
2. **Environment & Authentication**: Safely loads and sanitizes your Hugging Face credentials.
3. **QLoRA Fine-Tuning Pipeline**: Loads `Qwen/Qwen2.5-3B-Instruct` in 4-bit NormalFloat (NF4), applies LoRA adapters targeting all attention projection blocks, and trains on `mlabonne/guanaco-llama2-1k` via TRL's `SFTTrainer`.
4. **Interactive In-Notebook Evaluation**: Directly generates responses with the fine-tuned adapter using pure PyTorch, inspects output artifacts, and provides an optional export utility to download or push to the Hugging Face Hub.


---
## 1. Hardware Inspection & Dependency Setup

This notebook is optimized for consumer GPUs and cloud environments (NVIDIA T4, L4, A100, RTX 30/40 series).

> [!NOTE]
> **No vLLM Dependency**:
> Because this notebook focuses strictly on fine-tuning and PyTorch evaluation, it does **not** install `vllm`. This completely eliminates CUDA runtime version conflicts (such as `libcudart.so.13` mismatches) and keeps dependency installation fast (~30 seconds).

> [!IMPORTANT]
> **Transformers Version Pinning**:
> `transformers` is pinned to `<5.0.0` (`transformers>=4.45.0,<5.0.0`) to avoid breaking API changes introduced in Transformers v5 (e.g. deprecation of `all_special_tokens_extended` in tokenizers).


In [ ]:
!nvidia-smi


In [ ]:
# Install core fine-tuning dependencies
!pip install -q \
    "torch>=2.4.0" \
    "transformers>=4.45.0,<5.0.0" \
    "datasets>=3.0.0" \
    "trl>=0.11.0" \
    "peft>=0.13.0" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=1.0.0" \
    "python-dotenv"


---
## 2. Authentication & Environment Configuration

`Qwen/Qwen2.5-3B-Instruct` is ungated and open (Apache 2.0). A Hugging Face token is optional, but providing one avoids rate limiting when downloading weights and datasets, and enables pushing your trained adapter to the Hub.

> [!TIP]
> Always sanitize token strings by removing whitespace and Windows CRLF carriage returns (`\r`) to prevent HTTP header validation failures (`requests.exceptions.InvalidHeader`).


In [ ]:
import os
import getpass

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    print("Optional: Enter your Hugging Face Token (press Enter to skip):")
    entered_token = getpass.getpass("HF Token: ")
    if entered_token.strip():
        hf_token = entered_token.strip()

if hf_token:
    # Proactive sanitization against whitespace and carriage returns
    os.environ["HF_TOKEN"] = hf_token.strip()
    print("✓ HF_TOKEN loaded and sanitized.")
else:
    print("✓ Running unauthenticated (valid for open Qwen2.5 weights).")


---
## 3. QLoRA Fine-Tuning Pipeline (`app.py`)

### What makes QLoRA efficient?
1. **4-bit NormalFloat (NF4)**: An information-theoretically optimal quantile quantization scheme for normally distributed model weights.
2. **Double Quantization (DQ)**: Quantizes the quantization constants themselves, saving ~0.37 bits per parameter (~300MB on a 3B model).
3. **Paged Optimizers (`paged_adamw_8bit`)**: Allocates page-locked memory and automatically offloads optimizer state spikes to CPU RAM during memory pressure, preventing sudden out-of-memory (OOM) errors.
4. **LoRA Adapters**: Freezes all 3 billion base parameters and injects low-rank trainable decomposition matrices (`r=16`, `lora_alpha=32`) into attention blocks (`q_proj`, `v_proj`, etc.). Only the ~60MB adapter is trained and saved.


In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

# Configuration
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DATASET_ID = "mlabonne/guanaco-llama2-1k"
OUTPUT_DIR = "./qlora-adapter-output"

print(f"1. Loading tokenizer for: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit NormalFloat Quantization Config (bitsandbytes)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

print("2. Loading base model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

# Prepare model for k-bit training and disable KV cache for gradient checkpointing
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# PEFT LoRA Config targeting standard linear attention and MLP projections
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# Load training dataset
print(f"3. Loading dataset: {DATASET_ID}...")
dataset = load_dataset(DATASET_ID, split="train")
print(f"✓ Dataset loaded: {len(dataset)} instruction samples.")


In [ ]:
# Configure Supervised Fine-Tuning (SFT) hyperparameters
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=1024,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch size = 2 * 4 = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",       # Prevents OOM memory spikes
    gradient_checkpointing=True,    # Drastically cuts activation VRAM
    report_to="none",
    num_train_epochs=1,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print("Starting QLoRA fine-tuning...")
trainer.train()

print(f"Saving LoRA adapter weights to: {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✓ Training complete! LoRA adapter weights saved.")


---
## 4. In-Notebook Interactive Evaluation & Export

With fine-tuning complete, we evaluate the adapter directly using pure PyTorch and Hugging Face Transformers.

This step allows you to:
1. Validate generation quality with custom prompts.
2. Inspect the saved adapter weights and metadata.
3. (Optional) Export/Download the trained adapter folder or push to Hugging Face Hub.


In [ ]:
from transformers import TextStreamer

# 1. Clean up trainer and free activation memory
if "trainer" in locals():
    del trainer
torch.cuda.empty_cache()

# 2. Switch model to evaluation mode (disables LoRA dropout during inference)
model.eval()

# 3. Re-enable KV-cache for fast autoregressive generation (was disabled for gradient checkpointing)
model.config.use_cache = True

def generate_response(
    prompt: str,
    max_new_tokens: int = 512,
    temperature: float = 0.7,
    top_p: float = 0.9,
    repetition_penalty: float = 1.15,
    stream: bool = True,
) -> str:
    """Generates a response with the fine-tuned adapter using KV-caching, repetition penalty, and real-time streaming."""
    messages = [{"role": "user", "content": prompt}]
    formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_input, return_tensors="pt").to(model.device)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True) if stream else None

    # Handle all termination tokens (<|im_end|> and <|endoftext|>)
    eos_token_ids = [tokenizer.eos_token_id]
    endoftext_id = tokenizer.convert_tokens_to_ids("<|endoftext|>")
    if isinstance(endoftext_id, int) and endoftext_id not in eos_token_ids:
        eos_token_ids.append(endoftext_id)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=eos_token_ids,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram_size=3,
            streamer=streamer,
        )

    if not stream:
        generated_tokens = output_ids[0][inputs.input_ids.shape[1]:]
        return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return ""

test_prompt = "What are the core advantages of QLoRA fine-tuning?"
print(f"Prompt: {test_prompt}\n")
print("=" * 60)
print("Generated Response (Streaming):")
print("=" * 60)
generate_response(test_prompt, max_new_tokens=256, stream=True)


In [ ]:
# Interactive Test: Try your own prompt with full length (512 tokens)!
user_prompt = "Explain quantum computing in simple terms for a high school student."
print(f"User Prompt: {user_prompt}\n")
print("=" * 60)
print("Generated Response (Live Streaming):")
print("=" * 60)
generate_response(user_prompt, max_new_tokens=512, stream=True)


In [ ]:
import os

print(f"Inspecting saved adapter files in '{OUTPUT_DIR}':\n")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  - {fname:<30} ({size_mb:.2f} MB)")


---
### Optional: Download or Export Your Adapter

Run the cell below to package your fine-tuned adapter into a `.zip` archive for local use, or push it directly to the Hugging Face Hub.


In [ ]:
import shutil

# 1. Create a zip archive of the adapter
archive_name = "qlora_adapter_weights"
shutil.make_archive(archive_name, 'zip', OUTPUT_DIR)
print(f"✓ Created archive: {archive_name}.zip")

# 2. In Google Colab, uncomment the lines below to download directly to your machine:
# from google.colab import files
# files.download(f"{archive_name}.zip")

# 3. To push to Hugging Face Hub (requires HF_TOKEN with write permissions):
# target_repo_id = "your-username/qwen2.5-3b-qlora-guanaco"
# model.push_to_hub(target_repo_id)
# tokenizer.push_to_hub(target_repo_id)
# print(f"✓ Pushed adapter to https://huggingface.co/{target_repo_id}")


---
## 5. Troubleshooting & Engineering Notes

### 1. `AttributeError: Qwen2Tokenizer has no attribute all_special_tokens_extended`
- **Cause**: Installing an unconstrained `transformers>=4.45.0` can pull Transformers v5. Hugging Face removed `all_special_tokens_extended` in v5.
- **Solution**: Pin `transformers>=4.45.0,<5.0.0`.

### 2. `requests.exceptions.InvalidHeader: Invalid leading whitespace, reserved character(s)`
- **Cause**: Windows line endings (`\r\n`) in `.env` files or pasted tokens attach `\r` to `$HF_TOKEN`.
- **Solution**: Sanitize strings with `.strip()` before assigning to `os.environ["HF_TOKEN"]`.

### 3. Out-Of-Memory (OOM) During Training
- **Cause**: Large batch sizes or long sequence lengths exceeding GPU VRAM.
- **Solution**:
  - Keep `per_device_train_batch_size=2` and use `gradient_accumulation_steps=4`.
  - Ensure `optim="paged_adamw_8bit"` is enabled.
  - Keep `gradient_checkpointing=True` to recompute intermediate activations on the fly.
  - Set `max_length=1024` (or 512 for GPUs with <6GB VRAM).

### 4. Incomplete or Truncated Generation Output During Evaluation
- **Cause**:
  1. `max_new_tokens` was set too low (e.g., 150 tokens is only ~100 words; detailed explanations cut off mid-thought).
  2. `model.config.use_cache` remained `False` from training (disabled for gradient checkpointing), causing slow $O(N^2)$ generation.
  3. `model.eval()` was not called, leaving LoRA dropout (`0.05`) active during inference.
- **Solution**:
  - Switch to evaluation mode (`model.eval()`) and re-enable the KV-cache (`model.config.use_cache = True`).
  - Set `max_new_tokens=512` so responses have ample room to complete naturally.
  - Use `TextStreamer` from `transformers` to stream tokens to the console in real-time.
